# 01. Data Collection

## Dependencies

In [ ]:
!pip install -q datasets huggingface_hub pandas numpy matplotlib seaborn

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, Dataset, DatasetDict
from huggingface_hub import login
from google.colab import drive, userdata

# Setup

## Mount Drive & Project Directories

In [ ]:
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Vietnamese_HSD_Project'
DATA_RAW    = f'{PROJECT_DIR}/data/raw'
DATA_PROCESSED = f'{PROJECT_DIR}/data/processed'
DATA_AUGMENTED = f'{PROJECT_DIR}/data/augmented'
LOGS_DIR    = f'{PROJECT_DIR}/logs'
MODELS_DIR  = f'{PROJECT_DIR}/models'

for d in [DATA_RAW, DATA_PROCESSED, DATA_AUGMENTED, LOGS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Mount successfully")
print(f"Project dir: {PROJECT_DIR}")

## HuggingFace Login

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Successful login to HuggingFace!")

# Dataset

## Dataset 1: ViHSD

In [ ]:
print("Download ViHSD...")
vihsd_raw = load_dataset('uitnlp/vihsd')

print("\nStructures of ViHSD:")
print(vihsd_raw)

df_vihsd_train = vihsd_raw['train'].to_pandas()
df_vihsd_dev   = vihsd_raw['validation'].to_pandas()
df_vihsd_test  = vihsd_raw['test'].to_pandas()

print("\nData sample:")
print(df_vihsd_train.head(3))
print(f"\nSize: train={len(df_vihsd_train)}, dev={len(df_vihsd_dev)}, test={len(df_vihsd_test)}")
print("\nLabel distribution (train):")
print(df_vihsd_train['label_id'].value_counts())

In [ ]:
LABEL_MAP_VIHSD = {0: 'CLEAN', 1: 'OFFENSIVE', 2: 'HATE'}

def normalize_vihsd(df, split_name):
    out = pd.DataFrame()
    out['text']   = df['free_text']
    out['label']  = df['label_id'].map(LABEL_MAP_VIHSD)
    out['source'] = 'ViHSD'
    out['split']  = split_name
    return out

vihsd_train_norm = normalize_vihsd(df_vihsd_train, 'train')
vihsd_dev_norm   = normalize_vihsd(df_vihsd_dev, 'dev')
vihsd_test_norm  = normalize_vihsd(df_vihsd_test, 'test')

vihsd_train_norm.to_csv(f'{DATA_RAW}/vihsd_train.csv', index=False)
vihsd_dev_norm.to_csv(f'{DATA_RAW}/vihsd_dev.csv', index=False)
vihsd_test_norm.to_csv(f'{DATA_RAW}/vihsd_test.csv', index=False)

print("ViHSD saved to Drive")
print(vihsd_train_norm['label'].value_counts())

## Dataset 2: HSD-VLSP 2019

In [ ]:
print("Downloading HSD-VLSP...")

VLSP_FILE_PATH = "..."

df_vlsp = pd.read_csv(VLSP_FILE_PATH)

print(f"\nSize: {len(df_vlsp)} sample")
print("\nColumns: ")

In [ ]:
def normalize_vlsp(df):
    label_map = {0: 'CLEAN', 1: 'OFFENSIVE', 2: 'HATE'}
    df = df.copy()

    if 'sentence' in df.columns:
        df = df.rename(columns={'sentence': 'text'})
    if 'label_id' in df.columns:
        df = df.rename(columns={'label_id': 'label'})

    if pd.api.types.is_numeric_dtype(df['label']):
        df['label'] = df['label'].map(label_map)

    df['source'] = 'HSD-VLSP'
    df['split'] = 'train'

    return df[['text', 'label', 'source', 'split']]

vlsp_norm = normalize_vlsp(df_vlsp)
output_path = f'{DATA_RAW}/vlsp_train.csv'
vlsp_norm.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"Saved {len(vlsp_norm):,} template to: {output_path}")
print("\nLabel distribution:")
print(vlsp_norm['label'].value_counts())

display(vlsp_norm.head(3))

## Dataset 3: VOZ-HSD (text corpus, no labels)

In [ ]:
print("Downloading VOZ-HSD (streaming mode)...")
print("Note: Only extracting TEXT — ignoring AI-generated labels (unreliable per authors)")

voz_stream = load_dataset("tarudesu/VOZ-HSD", split="train", streaming=True)

sample_row = next(iter(voz_stream))
print(f"\nActual columns: {list(sample_row.keys())}")
print(f"Sample row    : {sample_row}")

TEXT_COL = next(
    (k for k in sample_row.keys()
     if k in ["texts", "text", "comment", "sentence", "content"]),
    list(sample_row.keys())[0]
)
print(f"Using column  : '{TEXT_COL}'")

VOZ_SAMPLE_SIZE = 500_000
voz_texts = []

voz_stream = load_dataset("tarudesu/VOZ-HSD", split="train", streaming=True)

for i, row in enumerate(voz_stream):
    if i >= VOZ_SAMPLE_SIZE:
        break
    text = row.get(TEXT_COL, "").strip()
    if text:
        voz_texts.append({
            "text"  : text,
            "source": "VOZ-HSD",
        })
    if (i + 1) % 100_000 == 0:
        print(f"  Processed {i+1:,} / {VOZ_SAMPLE_SIZE:,} rows...")

df_voz = pd.DataFrame(voz_texts)
output_path = f"{DATA_RAW}/voz_corpus_500k.csv"
df_voz.to_csv(output_path, index=False, encoding="utf-8-sig")

print()
print(f"VOZ corpus saved: {len(df_voz):,} samples")
print(f"Output path     : {output_path}")
print(f"Columns         : {df_voz.columns.tolist()}")
print()
print("IMPORTANT: This corpus is for Back Translation source & LLM few-shot examples ONLY.")
print("           Do NOT use as labeled training data for the classifier.")

## Dataset 4: ViHOS (span-level reference)

In [ ]:
data_files = {
    "train": "https://huggingface.co/datasets/phusroyal/ViHOS/raw/main/train_span_extraction/train.csv",
    "validation": "https://huggingface.co/datasets/phusroyal/ViHOS/raw/main/train_span_extraction/dev.csv",
    "test": "https://huggingface.co/datasets/phusroyal/ViHOS/raw/main/test/test.csv",
}

# Load the dataset using data_files, specifying the type as 'csv'
vihos_raw = load_dataset("csv", data_files=data_files)

vihos = vihos_raw["train"].to_pandas()
vihos.to_csv(f"{DATA_RAW}/vihos.csv", index=False, encoding="utf-8-sig")
print(vihos.shape, vihos.columns.tolist())
print(vihos.head(3))

# EDA: Label Distribution

In [ ]:
COLORS = {'CLEAN': '#9FE1CB', 'OFFENSIVE': '#FAC775', 'HATE': '#F09595'}

frames = [vihsd_train_norm]
if vlsp_norm is not None:
    frames.append(vlsp_norm)
df_all = pd.concat(frames, ignore_index=True)

def plot_bar(ax, series, title):
    counts = series.value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=[COLORS.get(l, '#ccc') for l in counts.index],
                  edgecolor='white')
    ax.set_title(title)
    ax.set_ylabel('Count')
    for bar, (_, val) in zip(bars, counts.items()):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 50,
                f'{val:,} ({val/len(series)*100:.1f}%)',
                ha='center', fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Vietnamese HSD — Label Distribution', fontsize=14, fontweight='bold')

plot_bar(axes[0], df_all['label'],           f'All (n={len(df_all):,})')
plot_bar(axes[1], vihsd_train_norm['label'], f'ViHSD (n={len(vihsd_train_norm):,})')

if vlsp_norm is not None:
    plot_bar(axes[2], vlsp_norm['label'], f'HSD-VLSP (n={len(vlsp_norm):,})')
else:
    axes[2].text(0.5, 0.5, 'HSD-VLSP not available',
                 ha='center', va='center', transform=axes[2].transAxes, color='gray')
    axes[2].axis('off')

plt.tight_layout()
plt.savefig(f'{LOGS_DIR}/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

c = df_all['label'].value_counts()
print(f"HATE:CLEAN = 1:{c['CLEAN']//c['HATE']}")

In [ ]:
df_all['word_count'] = df_all['text'].fillna('').apply(lambda x: len(x.split()))
df_all['char_count'] = df_all['text'].fillna('').apply(len)

print("="*40)
print(" " * 5 + "Text length statistics by label")
print("="*40)
print(df_all.groupby('label')[['word_count', 'char_count']].agg(['mean','median','max']))

print("="*40)
print(" " * 8 + "Statistics by data source")
print("="*40)
print(df_all.groupby('source')['word_count'].agg(['mean','median']))

# Split: Train / Dev / Test

In [ ]:
TEST_SET = vihsd_test_norm.copy()
DEV_SET  = vihsd_dev_norm.copy()

train_frames = [vihsd_train_norm]
if vlsp_norm is not None:
    train_frames.append(vlsp_norm)

TRAIN_SET = pd.concat(train_frames, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"train={len(TRAIN_SET):,} | dev={len(DEV_SET):,} | test={len(TEST_SET):,}")
print(TRAIN_SET['label'].value_counts())
print(f"imbalance HATE:CLEAN = 1:{TRAIN_SET['label'].value_counts()['CLEAN'] // TRAIN_SET['label'].value_counts()['HATE']}")

TRAIN_SET.to_csv(f'{DATA_PROCESSED}/train.csv', index=False, encoding='utf-8-sig')
DEV_SET.to_csv(  f'{DATA_PROCESSED}/dev.csv',   index=False, encoding='utf-8-sig')
TEST_SET.to_csv( f'{DATA_PROCESSED}/test.csv',  index=False, encoding='utf-8-sig')

# Export: Push to HuggingFace Hub

In [ ]:
from datasets import Dataset, DatasetDict

HF_USERNAME = 'HoaiAn001'
REPO_ID     = f'{HF_USERNAME}/vietnamese-hsd-combined'

hf_dataset = DatasetDict({
    'train'     : Dataset.from_pandas(TRAIN_SET.reset_index(drop=True)),
    'validation': Dataset.from_pandas(DEV_SET.reset_index(drop=True)),
    'test'      : Dataset.from_pandas(TEST_SET.reset_index(drop=True)),
})

hf_dataset.push_to_hub(REPO_ID, private=False, token=hf_token)
print(f"https://huggingface.co/datasets/{REPO_ID}")

In [ ]:
VOZ_REPO_ID = f'{HF_USERNAME}/voz-text-corpus-500k'

DatasetDict({'train': Dataset.from_pandas(df_voz)}).push_to_hub(
    VOZ_REPO_ID, private=True, token=hf_token
)
print(f"https://huggingface.co/datasets/{VOZ_REPO_ID}")

# Summary
- train / dev / test saved to `data/processed/`
- VOZ corpus (text only) saved to `data/raw/`
- datasets pushed to HuggingFace Hub
- next: `02_back_translation.ipynb`